In [7]:
import pandas as pd
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")
movies = movies.merge(credits, on="title")
movies.shape

(4809, 23)

In [9]:
movies = movies[["movie_id", "title", "overview", "genres", "keywords", "cast", "crew"]]
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [15]:
movies = movies.dropna()
movies = movies.drop_duplicates()
movies = movies.reset_index(drop=True)
len(movies)

4806

In [17]:
import ast 
def names(text):
    return [d["name"] for d in ast.literal_eval(text)]
def director(text):
    return [d["name"] for d in ast.literal_eval(text)
            if d["job"] == "Director"]
movies["genres"] = movies["genres"].apply(names)
movies["keywords"] = movies["keywords"].apply(names)
movies["cast"] = movies["cast"].apply(
    lambda x: names(x)[:3])
movies["crew"] = movies["crew"].apply(director)
movies.iloc[0]

movie_id                                                19995
title                                                  Avatar
overview    In the 22nd century, a paraplegic Marine is di...
genres          [Action, Adventure, Fantasy, Science Fiction]
keywords    [culture clash, future, space war, space colon...
cast         [Sam Worthington, Zoe Saldana, Sigourney Weaver]
crew                                          [James Cameron]
Name: 0, dtype: object

In [25]:
movies["overview"] = movies["overview"].apply(
    lambda x: x.split() if isinstance(x, str) else x)
for col in ["genres", "keywords", "cast", "crew"]:
    movies[col] = movies[col].apply(
        lambda x: [i.replace(" ", "") for i in x])
movies["tags"] = (
    movies["overview"]
    + movies["genres"]
    + movies["keywords"]
    + movies["cast"]
    + movies["crew"])

In [27]:
movies[["overview", "genres", "tags"]].head()

,overview,genres,tags
0,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[In, the, 22nd, century,, a, paraplegic, Marin..."
1,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[Captain, Barbossa,, long, believed, to, be, d..."
2,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[A, cryptic, message, from, Bond’s, past, send..."
3,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[Following, the, death, of, District, Attorney..."
4,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[John, Carter, is, a, war-weary,, former, mili..."


In [29]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
movies["tags"] = movies["tags"].apply(
    lambda x: " ".join(x).lower())
movies["tags"] = movies["tags"].apply(
    lambda x: " ".join(ps.stem(w) for w in x.split()))
print(ps.stem("loving"), ps.stem("loved"))

love love


In [31]:
from sklearn.feature_extraction.text import (
    CountVectorizer)
cs = CountVectorizer(max_features=5000,
                     stop_words="english")
vectors = cs.fit_transform(movies["tags"]).toarray()
vectors.shape

(4806, 5000)

In [37]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)
similarity.shape

(4806, 4806)

In [55]:
def recommend(movie):
    index = movies[movies["title"] == movie].index[0]
    scores = list(enumerate(similarity[index]))
    scores = sorted(scores, key=lambda x: x[1],
                    reverse=True)
    for i, score in scores[1:6]:
        print(movies.iloc[i].title)

In [57]:
recommend("Avatar")

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [61]:
recommend("Man of Steel")

Superman Returns
Batman v Superman: Dawn of Justice
Superman II
X-Men: First Class
Ong Bak 2


In [63]:
recommend ("Iron Man 3")

Iron Man
Iron Man 2
Avengers: Age of Ultron
Captain America: Civil War
X-Men


In [65]:
recommend("WALL·E")

Space Battleship Yamato
Meet Dave
Battlefield Earth
The R.M.
The Good Dinosaur


In [67]:
recommend("The Hunger Games")

The Hunger Games: Catching Fire
The Hunger Games: Mockingjay - Part 2
The Hunger Games: Mockingjay - Part 1
Indie Game: The Movie
The Indian in the Cupboard


In [69]:
recommend("Memoirs of a Geisha")

Letters from Iwo Jima
Tora! Tora! Tora!
Windtalkers
The Railway Man
Red Tails


In [71]:
recommend("Spider-Man")

Spider-Man 3
Spider-Man 2
The Amazing Spider-Man 2
Arachnophobia
Kick-Ass
